# PFEM/Transolver Training -- B1 x {NH, MR, AB}, B2 x {NH, MR, AB}
**Colab GPU runtime required (Runtime > Change runtime type > GPU)**

Runs the full PFEM pipeline (Wang et al., "Pretrain finite element method",
JMPS 214 (2026) 106682) for all 6 benchmark cases: B1 (unit square,
top-edge traction, fixed bottom) and B2 (quarter ring, R_in=1/R_out=2,
internal pressure, symmetry BCs), each with three hyperelastic energy
densities (Neo-Hookean, Mooney-Rivlin, Arruda-Boyce).

This notebook runs in two phases, per the advisor's guidance that blindly
training all 6 cases for 10000 epochs each is neither necessary nor
efficient (validation was too sparse to tell, and the best result so far
came around epoch 1000, with later epochs possibly hurt by optimization
instability or the LR schedule rather than genuine underfitting):

- **Phase 1 (screening)**: on one representative case (B1 x Neo-Hookean),
  screen a few real mini-batch sizes (a true parallel batch over samples
  sharing the same mesh, not gradient accumulation) for a short, fixed
  epoch budget, validating every 25-50 epochs and tracking wall-clock
  time, GPU peak memory, and optimizer steps alongside validation error.
  The most promising batch size is then continued to a larger epoch
  budget with early stopping (on validation error, not a fixed epoch
  count) -- producing a small comparison table and one concrete,
  automatically-chosen training protocol.
- **Phase 2 (the other 5 cases)**: that same protocol (batch size,
  validation cadence, early-stopping patience) is applied uniformly to
  the remaining cases -- no manual per-case tuning, per the advisor's
  explicit goal of "a robust and efficient procedure," not one-off tuning.

For each case, the pipeline is:
1. Generate a FEM ground-truth dataset (Total-Lagrangian Newton-Raphson,
   Q4 elements) via `omar_pfem/data/data_generate_B{1,2}.py`, parallelized
   across CPU cores -- skipped if the dataset already exists on disk.
2. Convert it to the Transolver NPZ format via `convert_B{1,2}_quad.py`.
3. Train a physics-informed Transolver (`omar_pfem/train_B{1,2}.py`) by
   minimizing total potential energy (Pi = U - W, no labeled-data loss),
   with n_hidden=256, n_layers=4, n_heads=8, slice_num=128 (PFEM's own
   architecture defaults) -- batch size and epoch budget now come from
   Phase 1 instead of being fixed in advance.

**Scale note**: PFEM's own reference script defaults to ntrain=800,
ntest=200 -- generating that many samples per case is still a real cost
independent of the training-protocol question above:
- **Data generation**: measured on a 4-core CPU at the reference 21x21 Q4
  mesh, generating each sample (a 10-step Newton-Raphson solve) took
  ~8s/sample with 4 parallel workers -- so a 1000-sample dataset
  (NTRAIN+NTEST) is roughly 2-2.5 CPU-hours per case, ~14 hours for all 6.
  Colab's own CPU allocation may differ from this benchmark.

Given that, this will likely span multiple Colab sessions. That's expected
and handled: every case checkpoints its model periodically and **resumes
from its own latest checkpoint** if this notebook is re-run (whether
because the runtime disconnected mid-case, or because you're continuing
across multiple sessions) -- so re-running `Runtime > Run all` after a
disconnect always makes forward progress instead of starting over.
Finished cases (`model_final.pt` or an `EARLY_STOPPED` marker on disk) are
skipped entirely, finished datasets (an NPZ already on disk) are never
regenerated, and a completed Phase 1 (a saved `training_protocol.json`)
is never re-screened.

If you'd rather trade fidelity for a faster full pass, lower
`TARGET_SAMPLES` in the config cell below -- everything else adapts
automatically.


## Cell 1 - Install dependencies

Colab's preinstalled `torch` already has CUDA support -- it is deliberately
NOT reinstalled here (a bare `pip install torch` risks silently replacing
it with a CPU-only wheel). Only the packages PFEM's Transolver model and
this pipeline actually need on top of Colab's base image are installed:
`einops`/`timm` (Transolver architecture), `h5py` (FEM dataset storage),
`jax` (autodiff-derived PK1 stress/tangent for Mooney-Rivlin/Arruda-Boyce
in the FEM generator -- CPU-only use, small dense tensors, no GPU needed).

Uses `{sys.executable} -m pip` rather than bare `!pip` -- on some Colab
runtimes the shell's `pip` has been observed to resolve to a different
Python than the notebook kernel itself, which makes packages "install
successfully" yet still fail to import.


In [ ]:
import sys
!{sys.executable} -m pip install -q einops timm h5py jax tqdm
print('Done - continue to Cell 2')


## Cell 2 - Clone the repo

If the repo is private, paste a GitHub personal access token as the value
of `GITHUB_TOKEN` below. Leave it as `""` if the repo is public -- never
type a literal `<TOKEN>` placeholder into the URL (`<`/`>` are bash
redirection operators and will break the clone before git even runs).

Always does a clean re-clone of the CODE (removes any previous
`/content/OMAR` first) so a broken partial clone can't linger -- but
generated datasets and results live elsewhere (local disk / Google Drive,
see Cells 3-4), **outside** the cloned repo, so re-cloning never discards
training progress.


In [ ]:
import os
import shutil
import sys

os.chdir('/content')

GITHUB_TOKEN = ""  # <-- paste your token between the quotes if the repo is private; leave "" if public
BRANCH = "claude/claude-code-question-d307wp"
REPO_URL = (f"https://{GITHUB_TOKEN}@github.com/suhibamro/omar.git" if GITHUB_TOKEN
            else "https://github.com/suhibamro/omar.git")

if os.path.exists('/content/OMAR'):
    shutil.rmtree('/content/OMAR')

!git clone -b {BRANCH} {REPO_URL} /content/OMAR

WORK_DIR = '/content/OMAR/Practical_Examples'
if not os.path.isdir(WORK_DIR):
    raise SystemExit(
        'ERROR: clone failed -- /content/OMAR/Practical_Examples does not exist.\n'
        'Scroll up to the "git clone" output above for the actual error.\n'
        'Common cause: the repo is private and GITHUB_TOKEN is still "".'
    )

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

pfem_ok = os.path.isdir(os.path.join(WORK_DIR, 'omar_pfem'))
if pfem_ok:
    print('Clone OK: omar_pfem/ found under Practical_Examples/.')
else:
    raise SystemExit('ERROR: omar_pfem/ not found -- clone looks incomplete or wrong branch.')

import torch
print(f'torch {torch.__version__} | cuda available={torch.cuda.is_available()}', end=' ')
if torch.cuda.is_available():
    print(f'| device={torch.cuda.get_device_name(0)}')
else:
    print()
    print('WARNING: no GPU detected -- go to Runtime > Change runtime type and select a GPU, '
          'then Runtime > Restart session and re-run from Cell 1.')


## Cell 3 - Mount Google Drive (persist training progress across full disconnects)

Colab's local `/content` disk only survives a *reconnect* to the same
runtime -- it does NOT survive a full runtime reset/reassignment (hitting
the session time limit, "Factory reset runtime", or Colab reclaiming an
idle VM). Given this pipeline can realistically run for many hours to
multiple days across several sessions, that's a real risk for the
expensive part of the run: GPU training checkpoints.

So results (`RESULTS_DIR` below) are written to Google Drive, which
survives any of the above. Every artifact this notebook produces ends up
under `RESULTS_DIR`: checkpoints, metrics, images, execution logs,
per-case manifests, and (see `ensure_dataset` in Cell 5) an archived copy
of each case's final NPZ dataset. Nothing that took real compute to
produce is left only on ephemeral local disk.

The one deliberate exception is the *scratch* work of FEM data generation
itself (`DATA_DIR`): the raw generation loop stays on local `/content`
disk, because Drive is FUSE-mounted (each file write is a network
round-trip) and the generator does hundreds of small sequential writes per
case -- on Drive that overhead would meaningfully slow down generation.
The moment generation finishes, its *result* (the converted NPZ) is copied
to `DATASETS_ARCHIVE_DIR` on Drive, so only the intermediate scratch work
is local and disposable, never the result. If prompted, click through the
Google auth flow -- this notebook will not work with your data without
that authorization.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted -- continue to Cell 4')


## Cell 4 - Configuration

Edit these to trade off dataset/training scale against wall-clock time.
`TARGET_SAMPLES` splits into `NTRAIN`/`NTEST` the same way as PFEM's own
reference script (800/200 by default -- lower this if a full 1000-sample
FEM generation pass is too slow on your Colab CPU allocation).

The screening-study settings (`SCREEN_*`) control Phase 1 only; its
outcome (`training_protocol.json`) then drives every case in Phase 2, so
there's nothing to configure per-case beyond dataset scale.


In [ ]:
import os

DATA_DIR = '/content/pfem_data'                                  # local disk: fast scratch during generation
RESULTS_DIR = '/content/drive/MyDrive/pfem_run/results'          # Google Drive: durable, holds every result
DATASETS_ARCHIVE_DIR = os.path.join(RESULTS_DIR, 'datasets')     # Google Drive: durable copy of each case's NPZ
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATASETS_ARCHIVE_DIR, exist_ok=True)

# ---- Dataset scale (PFEM's own reference: ntrain=800, ntest=200) ----
NTRAIN = 800
NTEST = 200
TARGET_SAMPLES = NTRAIN + NTEST   # total FEM samples generated per case

# ---- Mesh resolution (PFEM's own reference default for the beam case) ----
MESH_N = 21                      # Nx=Ny=21 for B1; Ntheta=Nr=21 for B2
GAUSS_POINTS_PER_ELEMENT = 4     # fixed by the FEM formulation: full 2x2 Gauss quadrature

# Reference cost of one native FE solve at this mesh resolution (10-step
# Newton-Raphson, 4-core CPU) -- the baseline the advisor asked training
# cost per sample to be compared against. Re-measure and update this if
# MESH_N changes.
NATIVE_FEM_SECONDS_PER_SAMPLE = 8.0

N_WORKERS = os.cpu_count()

# ---- Phase 1 (screening study on B1 x Neo-Hookean) ----
# Per the advisor's follow-up ("why stop at 16? continue until GPU memory
# becomes limiting or quality deteriorates"), the sweep no longer stops at
# a fixed upper size: screening_study.py tries these in ascending order
# and stops itself (reporting status="OOM" for the size that failed) the
# moment one doesn't fit on the GPU, rather than crashing the whole run --
# see screening_study.py's Phase 1 loop and is_oom_error().
SCREEN_BATCH_SIZES = "4,8,16,32,64,128,256"   # comma string, passed straight to screening_study.py
SCREEN_EPOCHS = 400             # short, fixed budget per batch size (advisor: ~250-500)
VALIDATE_EVERY = 25             # advisor: validate every 25-50 epochs
CONTINUE_EPOCHS = 2000          # upper bound for the winning config (advisor: ~1000-2000)
EARLY_STOP_PATIENCE = 8         # validation events with no improvement before stopping
EARLY_STOP_MIN_DELTA = 1e-4

GEOMETRIES = ["B1", "B2"]
MATERIALS = ["neo_hookean", "mooney_rivlin", "arruda_boyce"]
CASES = [(g, m) for g in GEOMETRIES for m in MATERIALS]
SCREENING_CASE = ("B1", "neo_hookean")

print(f'CPU workers for data generation: {N_WORKERS}')
print(f'Target samples/case: {TARGET_SAMPLES} (ntrain={NTRAIN}, ntest={NTEST})')
print(f'Screening case: {SCREENING_CASE}, batch_sizes={SCREEN_BATCH_SIZES}, '
      f'screen_epochs={SCREEN_EPOCHS}, validate_every={VALIDATE_EVERY}')
print(f'Cases: {CASES}')


## Cell 5 - Helpers: everything this notebook produces is saved to Drive

Five helpers, all built around one rule -- **nothing that took real compute
to produce is ever left only on ephemeral local disk**:

- `run_streaming(cmd, cwd, log_path=None)` runs a command as a subprocess
  and prints its stdout live (unbuffered, `python -u`). When `log_path` is
  given (always, below -- pointed at `RESULTS_DIR`), every line is *also*
  appended to that file, so the full raw execution log of every data
  generation, conversion, screening, and training step survives a Colab
  disconnect even if the notebook's own output pane does not.
- `ensure_dataset(geometry, material)` generates a case's FEM dataset on
  fast local scratch disk, then immediately archives the resulting NPZ to
  `DATASETS_ARCHIVE_DIR` on Drive; on a later call (even in a fresh
  session, after local disk was wiped) it restores the archived copy
  instead of regenerating from scratch. Data generation is CPU-hours of
  work -- it is treated exactly as durably as training results, not as
  disposable scratch.
- `count_trainable_params()` instantiates the Transolver model with the
  exact architecture hyperparameters Cell 7 trains with, and returns its
  trainable-parameter count -- computed once, from the real model, not a
  hand-typed number that could drift out of sync with the architecture.
- `mesh_facts(mesh_n)` returns the exact node/element/integration-point
  counts for a given mesh resolution (structured Q4 grid, full 2x2 Gauss
  quadrature), so the discretization reported anywhere is always derived
  from `MESH_N`, never hand-typed.
- `write_case_manifest(name, geometry, material)` writes one always-current
  JSON per case (`RESULTS_DIR/{case}/case_manifest.json`) collecting every
  quantity needed for the written report in one place: architecture,
  trainable parameters, discretization, dataset size, the training
  protocol used, and (once available) final validation error, wall-clock
  time, optimizer steps, and peak GPU memory (allocated and reserved).
  Overwritten every time a case's status is checked, so it always reflects
  the current state, including for a case that is still mid-training.
- `save_completion_snapshot(name)` additionally writes one permanent,
  timestamped record (`RESULTS_DIR/completed_runs/{case}_{timestamp}.json`)
  the first time a case is found finished -- never overwritten, so it is
  an audit trail of exactly when each case completed, independent of the
  always-current manifest above.


In [ ]:
import subprocess
import time
import json as _json
import datetime as _dt

def run_streaming(cmd, cwd=None, log_path=None):
    """Runs cmd, streaming its stdout live. If log_path is given (always,
    below), every line is also appended there -- log_path should point at
    a Drive-backed (RESULTS_DIR) location so the raw log survives a
    disconnect even when the notebook's own cell output does not."""
    print(f"$ {' '.join(cmd)}")
    log_f = open(log_path, "a") if log_path else None
    if log_f:
        log_f.write(f"\n===== {_dt.datetime.now().isoformat()} =====\n$ {' '.join(cmd)}\n")
        log_f.flush()
    proc = subprocess.Popen(
        cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
        if log_f:
            log_f.write(line)
    proc.wait()
    if log_f:
        log_f.write(f"[exit code {proc.returncode}] {_dt.datetime.now().isoformat()}\n")
        log_f.close()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(cmd)}")


def case_name(geometry, material):
    return f"{geometry}_{material}"


def case_log_path(name, step):
    log_dir = os.path.join(RESULTS_DIR, name, "logs")
    os.makedirs(log_dir, exist_ok=True)
    return os.path.join(log_dir, f"{step}.log")


_TRAINABLE_PARAMS_CACHE = {}

def count_trainable_params():
    """Instantiates the exact Transolver architecture Cell 7 trains with
    and counts its trainable parameters -- computed once and cached, so
    the number reported in every manifest is guaranteed to match the real
    model, not a hand-typed constant that could silently drift."""
    if "n" in _TRAINABLE_PARAMS_CACHE:
        return _TRAINABLE_PARAMS_CACHE["n"]
    from omar_pfem.model.Transolver_Irregular_Mesh import Model
    m = Model(space_dim=2, n_layers=4, n_hidden=256, dropout=0.1, n_head=8,
              Time_Input=False, mlp_ratio=2, fun_dim=4, out_dim=2,
              slice_num=128, ref=16, unified_pos=0)
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    _TRAINABLE_PARAMS_CACHE["n"] = n
    return n


def mesh_facts(mesh_n):
    """Structured Q4 mesh facts for an mesh_n x mesh_n node grid (B1's
    Nx=Ny or B2's Ntheta=Nr), full 2x2 Gauss quadrature."""
    n_nodes = mesh_n * mesh_n
    n_elements = (mesh_n - 1) * (mesh_n - 1)
    return {
        "nodes_per_side": mesh_n,
        "n_nodes": n_nodes,
        "n_elements": n_elements,
        "gauss_points_per_element": GAUSS_POINTS_PER_ELEMENT,
        "integration_points_per_sample": n_elements * GAUSS_POINTS_PER_ELEMENT,
    }


def write_case_manifest(name, geometry, material):
    """Writes/overwrites one always-current, comprehensive record for a
    case: architecture, discretization, dataset size, protocol, and (once
    available) final training outcome -- every quantity needed for the
    written report, in one place, always in sync with the real run."""
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(os.path.join(out_dir, "EARLY_STOPPED")):
        status = "EARLY_STOPPED"
    elif os.path.exists(os.path.join(out_dir, "model_final.pt")):
        status = "model_final"
    else:
        status = "in_progress"

    metrics_path = os.path.join(out_dir, "metrics_history.json")
    history = []
    if os.path.exists(metrics_path):
        with open(metrics_path) as f:
            history = _json.load(f)
    best_meta = {}
    best_meta_path = os.path.join(out_dir, "best_checkpoint_meta.json")
    if os.path.exists(best_meta_path):
        with open(best_meta_path) as f:
            best_meta = _json.load(f)

    inference_stats = {}
    inference_path = os.path.join(out_dir, "inference_latency.json")
    if os.path.exists(inference_path):
        with open(inference_path) as f:
            inference_stats = _json.load(f)

    final_rec = history[-1] if history else None
    protocol_used = PROTOCOL if "PROTOCOL" in globals() else None

    manifest = {
        "case": name,
        "geometry": geometry,
        "material": material,
        "status": status,
        "last_updated": _dt.datetime.now().isoformat(),
        "architecture": {
            "n_layers": 4, "n_hidden": 256, "n_heads": 8, "mlp_ratio": 2,
            "dropout": 0.1, "slice_num": 128, "fun_dim": 4,
            "trainable_parameters": count_trainable_params(),
        },
        "discretization": mesh_facts(MESH_N),
        "dataset": {
            "ntrain": NTRAIN, "ntest": NTEST, "total_samples": NTRAIN + NTEST,
            "native_fem_seconds_per_sample": NATIVE_FEM_SECONDS_PER_SAMPLE,
        },
        "protocol_used": protocol_used,
        "training_outcome": {
            "best_val_error": best_meta.get("best_val_error"),
            "best_epoch": best_meta.get("best_epoch"),
            "final_epoch": final_rec["epoch"] if final_rec else None,
            "num_validation_events": len(history),
            "total_wall_clock_s": final_rec["cumulative_wall_clock_s"] if final_rec else None,
            "total_opt_steps": final_rec["opt_steps"] if final_rec else None,
            "gpu_peak_mem_allocated_mb": max((r.get("gpu_peak_mem_mb", 0.0) for r in history), default=None),
            "gpu_peak_mem_reserved_mb": max((r.get("gpu_peak_mem_reserved_mb", 0.0) for r in history), default=None),
            # Whole-device usage (torch.cuda.mem_get_info(), nvidia-smi-like)
            # -- distinct from the two PyTorch-allocator figures above, which
            # exclude the CUDA context's own fixed overhead and any other
            # process sharing the GPU. See train_B1.py/train_B2.py.
            "gpu_peak_mem_device_mb": max((r.get("gpu_peak_mem_device_mb", 0.0) for r in history), default=None),
            "gpu_device_total_mb": max((r.get("gpu_device_total_mb", 0.0) for r in history), default=None),
            "training_cost_per_sample_epoch_s": (
                final_rec["cumulative_wall_clock_s"] / (final_rec["epoch"] * NTRAIN)
                if final_rec else None
            ),
            # Pure forward-pass latency (batch_size=1, no backward pass, no
            # energy assembly) measured once at the end of training -- see
            # benchmark_inference_latency_Q4 in train_B1.py/train_B2.py.
            "inference_ms_per_sample": inference_stats.get("inference_ms_per_sample"),
        },
    }
    manifest_path = os.path.join(out_dir, "case_manifest.json")
    with open(manifest_path, "w") as f:
        _json.dump(manifest, f, indent=2)
    return manifest


def save_completion_snapshot(name):
    """Writes one permanent timestamped summary the first time `name` is
    found finished. Safe to call every time a case is checked (skips if a
    snapshot already exists), so re-running the notebook never spams
    duplicates for cases that were already done."""
    out_dir = os.path.join(RESULTS_DIR, name)
    if os.path.exists(os.path.join(out_dir, "EARLY_STOPPED")):
        status = "EARLY_STOPPED"
    elif os.path.exists(os.path.join(out_dir, "model_final.pt")):
        status = "model_final"
    else:
        return  # not finished yet -- nothing permanent to record

    metrics_path = os.path.join(out_dir, "metrics_history.json")
    if not os.path.exists(metrics_path):
        return

    snap_dir = os.path.join(RESULTS_DIR, "completed_runs")
    os.makedirs(snap_dir, exist_ok=True)
    if any(f.startswith(name + "_") for f in os.listdir(snap_dir)):
        return  # already have a permanent record for this case

    with open(metrics_path) as f:
        history = _json.load(f)
    best_meta = {}
    best_meta_path = os.path.join(out_dir, "best_checkpoint_meta.json")
    if os.path.exists(best_meta_path):
        with open(best_meta_path) as f:
            best_meta = _json.load(f)

    timestamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    snapshot = {
        "case": name,
        "status": status,
        "snapshot_taken_at": timestamp,
        "best_checkpoint": best_meta,
        "last_validation_record": history[-1] if history else None,
        "num_validation_events": len(history),
    }
    snap_path = os.path.join(snap_dir, f"{name}_{timestamp}.json")
    with open(snap_path, "w") as f:
        _json.dump(snapshot, f, indent=2)
    print(f"[{name}] permanent completion snapshot saved -> {snap_path}")


def ensure_dataset(geometry, material):
    """Generates + converts the FEM dataset for one case on fast local
    scratch disk if it isn't already there, then archives the result to
    Drive (DATASETS_ARCHIVE_DIR) so it survives a full disk wipe. On a
    later call (this session or a fresh one), restores the archived copy
    instead of regenerating. Returns the (local) NPZ path."""
    name = case_name(geometry, material)
    h5_dir = os.path.join(DATA_DIR, f"fem_{name}")
    npz_dir = os.path.join(DATA_DIR, f"npz_{name}")
    npz_path = os.path.join(npz_dir, "hyperelastic_training_data_q4.npz")
    archive_path = os.path.join(DATASETS_ARCHIVE_DIR, name, "hyperelastic_training_data_q4.npz")

    if os.path.exists(npz_path):
        print(f"[{name}] NPZ dataset already exists locally at {npz_path}, skipping generation.")
        return npz_path

    if os.path.exists(archive_path):
        print(f"[{name}] Restoring archived NPZ dataset from Drive ({archive_path}) -- no regeneration needed.")
        os.makedirs(npz_dir, exist_ok=True)
        shutil.copy2(archive_path, npz_path)
        return npz_path

    t0 = time.time()
    if geometry == "B1":
        gen_module = "omar_pfem.data.data_generate_B1"
        gen_args = ["--Nx", str(MESH_N), "--Ny", str(MESH_N)]
        conv_module = "omar_pfem.data.convert_B1_quad"
    else:
        gen_module = "omar_pfem.data.data_generate_B2"
        gen_args = ["--Ntheta", str(MESH_N), "--Nr", str(MESH_N)]
        conv_module = "omar_pfem.data.convert_B2_quad"

    run_streaming([
        sys.executable, "-u", "-m", gen_module,
        "--num_index", "1", "--num_samples", str(TARGET_SAMPLES),
        *gen_args, "--material", material,
        "--n_workers", str(N_WORKERS),
        "--out_dir", h5_dir,
    ], cwd=WORK_DIR, log_path=case_log_path(name, "data_generation"))

    run_streaming([
        sys.executable, "-u", "-m", conv_module,
        "--h5_dir", h5_dir, "--out_dir", npz_dir,
    ], cwd=WORK_DIR, log_path=case_log_path(name, "npz_conversion"))

    os.makedirs(os.path.dirname(archive_path), exist_ok=True)
    shutil.copy2(npz_path, archive_path)
    print(f"[{name}] Dataset ready after {time.time()-t0:.0f}s -- archived to Drive at {archive_path}")
    return npz_path


## Cell 6 - Phase 1: screening study (B1 x Neo-Hookean)

Runs `omar_pfem/screening_study.py`, which:
1. Trains B1 x Neo-Hookean at each of `SCREEN_BATCH_SIZES` for a short,
   fixed `SCREEN_EPOCHS` budget, validating every `VALIDATE_EVERY` epochs
   and logging validation error, wall-clock time, GPU peak memory, and
   optimizer-step count for each.
2. Picks the batch size with the lowest best validation error and
   continues *that same run* (resumed from its own screening checkpoint)
   to `CONTINUE_EPOCHS`, this time with early stopping
   (`EARLY_STOP_PATIENCE` validation events with no improvement).

`--final_out_dir` points the continuation directly at this case's real
results directory, so it doubles as B1 x Neo-Hookean's actual training run
-- Cell 7's main loop later finds it already done and skips it, no
duplicated work.

If a previous run already finished Phase 1 (`training_protocol.json`
exists), this cell loads that instead of re-screening -- so a Colab
disconnect never repeats the screening study itself. To force a fresh
Phase 1 run (e.g. to regenerate a comparison table that was lost before
the Drive-persistence fix below), delete `training_protocol.json` from
`RESULTS_DIR` before running this cell -- Phase 2 will then write into
B1 x Neo-Hookean's real results directory exactly as it always has, so a
fresh run replaces whatever was there before.

The screening runs (`bs4`/`bs8`/`bs16` + the comparison table) are written
under `RESULTS_DIR` (Google Drive), not `DATA_DIR` (local disk): unlike the
FEM datasets, this comparison took real training compute to produce, so it
belongs with the other things that must survive a disconnect, not with the
data that's cheap to regenerate.


In [ ]:
protocol_path = os.path.join(RESULTS_DIR, "training_protocol.json")

if os.path.exists(protocol_path):
    with open(protocol_path) as f:
        PROTOCOL = _json.load(f)
    print(f"Phase 1 already completed -- loaded protocol from {protocol_path}:")
    print(_json.dumps(PROTOCOL, indent=2))
    # Safety check: if this protocol was screened with a DIFFERENT/smaller
    # SCREEN_BATCH_SIZES than the one configured now (e.g. it predates the
    # advisor's "continue until OOM" extension from {4,8,16} to
    # {4,8,16,32,64,128,256}), silently trusting it would mean the batch
    # sizes the advisor actually asked to see were never screened.
    # IMPORTANT: do NOT "fix" this by deleting training_protocol.json and
    # re-running this cell -- Phase 2 below writes directly into
    # B1 x Neo-Hookean's REAL results directory and would overwrite its
    # already-completed checkpoint with a fresh run at whatever the new
    # winning batch size turns out to be. Use the safe, separate
    # "extended batch-size screening" cell instead (produces the
    # comparison table in its own directory, touches nothing else).
    screened = PROTOCOL.get("screened_batch_sizes")
    if screened is not None and screened != SCREEN_BATCH_SIZES:
        print(f"\nWARNING: this protocol was screened with batch_sizes={screened}, "
              f"but SCREEN_BATCH_SIZES is now {SCREEN_BATCH_SIZES} -- the wider sweep "
              f"was never actually run. Do NOT delete {protocol_path} to force a "
              f"re-screen -- that would overwrite B1_neo_hookean's real, already-"
              f"completed training. Instead run the safe standalone extended-screening "
              f"cell (writes to RESULTS_DIR/screening_extended_{{case}}, touches nothing else).")
    elif screened is None:
        print(f"\nNote: this protocol predates batch-size tracking, so it's unknown which "
              f"batch_sizes it was screened with. If in doubt whether the full "
              f"SCREEN_BATCH_SIZES={SCREEN_BATCH_SIZES} sweep was ever run, use the safe "
              f"standalone extended-screening cell to check/produce the comparison table "
              f"without touching this protocol or B1_neo_hookean's real results.")
else:
    screen_geometry, screen_material = SCREENING_CASE
    screen_npz_path = ensure_dataset(screen_geometry, screen_material)
    screen_out_dir = os.path.join(RESULTS_DIR, "screening_" + case_name(*SCREENING_CASE))
    final_out_dir = os.path.join(RESULTS_DIR, case_name(*SCREENING_CASE))

    run_streaming([
        sys.executable, "-u", "-m", "omar_pfem.screening_study",
        "--path", screen_npz_path,
        "--geometry", screen_geometry, "--material", screen_material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--batch_sizes", SCREEN_BATCH_SIZES,
        "--screen_epochs", str(SCREEN_EPOCHS),
        "--validate_every", str(VALIDATE_EVERY),
        "--continue_epochs", str(CONTINUE_EPOCHS),
        "--continue_early_stop_patience", str(EARLY_STOP_PATIENCE),
        "--continue_early_stop_min_delta", str(EARLY_STOP_MIN_DELTA),
        "--final_out_dir", final_out_dir,
        "--out_dir", screen_out_dir,
    ], cwd=WORK_DIR, log_path=case_log_path(case_name(*SCREENING_CASE), "screening_driver"))

    with open(os.path.join(screen_out_dir, "screening_summary.json")) as f:
        screening_rows = _json.load(f)
    # Only rows that actually completed the screening budget (status=="OK")
    # are eligible to win -- a row with status=="OOM" or "FAILED" can still
    # have some validation history (it may have crashed after several
    # validation events, not on the very first one), so filtering on
    # best_val_error alone is not enough to exclude it. screening_study.py's
    # own Phase 2 selection uses this same filter internally.
    ok_rows = [r for r in screening_rows if r.get("status") == "OK" and r["best_val_error"] is not None]
    oom_rows = [r for r in screening_rows if r.get("status") == "OOM"]
    if oom_rows:
        print(f"Batch size(s) that ran out of GPU memory during screening: "
              f"{[r['batch_size'] for r in oom_rows]} -- excluded from winner selection.")
    winner_bs = min(ok_rows, key=lambda r: r["best_val_error"])["batch_size"]

    PROTOCOL = {
        "batch_size": winner_bs,
        "validate_every": VALIDATE_EVERY,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        "epochs": CONTINUE_EPOCHS,
        "screened_batch_sizes": SCREEN_BATCH_SIZES,
    }
    with open(protocol_path, "w") as f:
        _json.dump(PROTOCOL, f, indent=2)

    print("\nPhase 1 done. Screening summary:")
    with open(os.path.join(screen_out_dir, "screening_summary.md")) as f:
        print(f.read())
    print(f"\nChosen protocol (saved to {protocol_path}):")
    print(_json.dumps(PROTOCOL, indent=2))

write_case_manifest(case_name(*SCREENING_CASE), *SCREENING_CASE)
save_completion_snapshot(case_name(*SCREENING_CASE))


## Cell 6B - (fallback) Safe standalone extended batch-size screening

Only needed if Cell 6 printed a **WARNING** above that `training_protocol.json`
was screened with a narrower `batch_sizes` than currently configured (this
happens if Phase 1 completed before `SCREEN_BATCH_SIZES` was extended per
the advisor's "continue until OOM" request). Produces the real
batch-size-vs-memory-vs-cost-vs-quality comparison table in its **own**
directory -- does NOT touch `training_protocol.json` or
`RESULTS_DIR/B1_neo_hookean` (the real, already-completed case), so it is
always safe to run regardless of what Cell 6 decided. Skip this cell
entirely if Cell 6 did not print a warning.


In [ ]:
extended_screen_npz_path = ensure_dataset(*SCREENING_CASE)
extended_screen_out_dir = os.path.join(RESULTS_DIR, "screening_extended_" + case_name(*SCREENING_CASE))
os.makedirs(extended_screen_out_dir, exist_ok=True)

if os.path.exists(os.path.join(extended_screen_out_dir, "screening_summary.json")):
    print(f"Extended screening already done -> {extended_screen_out_dir}, skipping.")
else:
    run_streaming([
        sys.executable, "-u", "-m", "omar_pfem.screening_study",
        "--path", extended_screen_npz_path,
        "--geometry", SCREENING_CASE[0], "--material", SCREENING_CASE[1],
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--batch_sizes", SCREEN_BATCH_SIZES,
        "--screen_epochs", str(SCREEN_EPOCHS),
        "--validate_every", str(VALIDATE_EVERY),
        "--continue_epochs", str(CONTINUE_EPOCHS),
        "--continue_early_stop_patience", str(EARLY_STOP_PATIENCE),
        "--continue_early_stop_min_delta", str(EARLY_STOP_MIN_DELTA),
        "--out_dir", extended_screen_out_dir,
        # Deliberately NO --final_out_dir: keeps this fully separate from
        # RESULTS_DIR/B1_neo_hookean (the real, already-completed case).
    ], cwd=WORK_DIR, log_path=os.path.join(extended_screen_out_dir, "run.log"))

with open(os.path.join(extended_screen_out_dir, "screening_summary.md")) as f:
    print(f.read())


## Cell 7 - Run the remaining 5 cases with the chosen protocol

Applies `PROTOCOL` from Phase 1 uniformly -- no per-case tuning, per the
advisor's explicit goal. B1 x Neo-Hookean is skipped here since Phase 1
already trained it (its `model_final.pt`/`EARLY_STOPPED` marker is
already in place).

For every case (skipped or freshly trained), `write_case_manifest` updates
`RESULTS_DIR/{case}/case_manifest.json` with the current architecture,
discretization, dataset size, protocol, and training outcome, and the full
raw training log is appended to `RESULTS_DIR/{case}/logs/training.log`.


In [ ]:
def run_one_case(geometry, material):
    name = case_name(geometry, material)
    out_dir = os.path.join(RESULTS_DIR, name)

    print(f"\n{'='*80}\n===== CASE: {name} =====\n{'='*80}")

    if os.path.exists(os.path.join(out_dir, "model_final.pt")) or \
       os.path.exists(os.path.join(out_dir, "EARLY_STOPPED")):
        print(f"[{name}] already fully trained (model_final.pt or EARLY_STOPPED found), skipping.")
        write_case_manifest(name, geometry, material)
        save_completion_snapshot(name)
        return

    npz_path = ensure_dataset(geometry, material)

    train_module = "omar_pfem.train_B1" if geometry == "B1" else "omar_pfem.train_B2"
    t1 = time.time()
    run_streaming([
        sys.executable, "-u", "-m", train_module,
        "--path", npz_path,
        "--material", material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--epochs", str(PROTOCOL["epochs"]),
        "--batch_size", str(PROTOCOL["batch_size"]),
        "--validate_every", str(PROTOCOL["validate_every"]),
        "--save_every", str(PROTOCOL["validate_every"]),
        "--early_stop_patience", str(PROTOCOL["early_stop_patience"]),
        "--early_stop_min_delta", str(PROTOCOL["early_stop_min_delta"]),
        "--print_every", "999999",
        "--out_dir", out_dir,
    ], cwd=WORK_DIR, log_path=case_log_path(name, "training"))
    print(f"[{name}] Training finished after {time.time()-t1:.0f}s")
    write_case_manifest(name, geometry, material)
    save_completion_snapshot(name)


for geometry, material in CASES:
    if (geometry, material) == SCREENING_CASE:
        continue  # already trained as part of Phase 1
    run_one_case(geometry, material)

print("\nAll cases either completed or already were -- see per-case status above.")


## Cell 7B - Backfill inference-latency + GPU-memory profile for already-trained cases

Only matters for cases that were ALREADY fully trained (Cell 6/7 above
printed "already fully trained ... skipping" for them) **before**
`benchmark_inference_latency_Q4` and the device-level GPU-memory
instrumentation existed in this codebase. Both live inside
`train_hyperelastic_Q4()`, right after the main training loop -- but that
function returns EARLY, before ever reaching them, whenever a case is
already done. So for a case finished in an earlier phase of this project,
simply re-running Cell 6/7 never produces `inference_latency.json` or the
`gpu_peak_mem_device_mb`/`gpu_device_total_mb` fields, even though the
code to compute them now exists. If Cell 6/7 above trained every case
fresh in THIS run, this cell will find nothing to backfill and do nothing.

This cell fixes both, safely:
1. **Inference latency**: `measure_inference_latency.py` loads the
   existing checkpoint and re-derives the exact same measurement
   standalone (no retraining), writing `inference_latency.json` into the
   case's own directory -- purely additive, never touches
   `model_final.pt` or `metrics_history.json`.
2. **GPU memory**: this genuinely requires a real training pass to
   measure (it's not something you can compute from a static checkpoint),
   so this runs a SHORT, throwaway training run (3 epochs, same batch
   size/architecture as the real protocol) to a SEPARATE directory under
   `memory_profile_reruns/` -- never touches the real case's files. Read
   `gpu_peak_mem_device_mb`/`gpu_device_total_mb` from that throwaway
   run's own `metrics_history.json` afterward.


In [ ]:
MEMORY_PROFILE_EPOCHS = 3   # short + throwaway -- just enough to observe real peak GPU memory
memory_profile_dir = os.path.join(RESULTS_DIR, "memory_profile_reruns")
os.makedirs(memory_profile_dir, exist_ok=True)

for geometry, material in CASES:
    name = case_name(geometry, material)
    case_dir = os.path.join(RESULTS_DIR, name)
    checkpoint = os.path.join(case_dir, "model_best.pt")
    if not os.path.exists(checkpoint):
        checkpoint = os.path.join(case_dir, "model_final.pt")
    if not os.path.exists(checkpoint):
        print(f"[{name}] no checkpoint yet -- run Cell 6/7 first, skipping backfill.")
        continue
    dataset_path = os.path.join(DATASETS_ARCHIVE_DIR, name, "hyperelastic_training_data_q4.npz")

    # 1) Inference latency: safe, additive, no retraining.
    inf_json = os.path.join(case_dir, "inference_latency.json")
    if os.path.exists(inf_json):
        print(f"[{name}] inference_latency.json already present, skipping.")
    else:
        inf_log_dir = os.path.join(case_dir, "logs")
        os.makedirs(inf_log_dir, exist_ok=True)
        run_streaming([
            sys.executable, "-u", "-m", "omar_pfem.measure_inference_latency",
            "--geometry", geometry, "--material", material,
            "--checkpoint", checkpoint, "--dataset", dataset_path,
            "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
            "--out_json", inf_json,
        ], cwd=WORK_DIR, log_path=os.path.join(inf_log_dir, "inference_latency_backfill.log"))

    # 2) GPU memory profile: short, throwaway training run to a SEPARATE
    # directory -- never touches the real case's model_final.pt/metrics.
    profile_out = os.path.join(memory_profile_dir, name)
    os.makedirs(profile_out, exist_ok=True)
    profile_metrics = os.path.join(profile_out, "metrics_history.json")
    if os.path.exists(profile_metrics):
        print(f"[{name}] memory profile already exists -> {profile_metrics}, skipping.")
        continue
    train_module = "omar_pfem.train_B1" if geometry == "B1" else "omar_pfem.train_B2"
    run_streaming([
        sys.executable, "-u", "-m", train_module,
        "--path", dataset_path, "--material", material,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST),
        "--batch_size", str(PROTOCOL["batch_size"]),
        "--epochs", str(MEMORY_PROFILE_EPOCHS), "--validate_every", "1", "--save_every", "1",
        "--early_stop_patience", "0", "--print_every", "999999",
        "--out_dir", profile_out,
    ], cwd=WORK_DIR, log_path=os.path.join(profile_out, "run.log"))

print("\nBackfill pass complete.")
print("Inference latency: written directly into each case's own directory (RESULTS_DIR/{case}/inference_latency.json).")
print(f"GPU memory profiles ({MEMORY_PROFILE_EPOCHS}-epoch throwaway runs at the real batch size): "
      f"see {memory_profile_dir}/{{case}}/metrics_history.json for gpu_peak_mem_device_mb / gpu_device_total_mb.")


## Cell 8 - Summary: metrics + loss curves across all 6 cases

Reads each case's `metrics_history.json` -- works even for cases that are
still mid-run or were only partially completed before a disconnect.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
summary_rows = []

for ax, (geometry, material) in zip(axes.flat, CASES):
    name = case_name(geometry, material)
    metrics_path = os.path.join(RESULTS_DIR, name, "metrics_history.json")
    if not os.path.exists(metrics_path):
        ax.set_title(f"{name}\n(no data yet)")
        continue

    with open(metrics_path) as f:
        history = _json.load(f)
    if not history:
        ax.set_title(f"{name}\n(empty history)")
        continue

    epochs = [h["epoch"] for h in history]
    val_err = [h["val_error"] for h in history]

    ax.semilogy(epochs, val_err, label="val_error")
    ax.set_title(f"{name} (epoch {epochs[-1]})")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

    summary_rows.append((name, epochs[-1], history[-1]["mean_rel_L2_u"], history[-1]["mean_rel_L2_v"]))

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "all_cases_loss_curves.png"), dpi=150)
plt.show()

print(f"{'case':<20s} {'last epoch':>10s} {'RelL2(u)':>12s} {'RelL2(v)':>12s}")
for name, ep, u, v in summary_rows:
    print(f"{name:<20s} {ep:>10d} {u:>12.3e} {v:>12.3e}")


## Cell 9 - Mesh (h-refinement) convergence study for the reference FE solver

Per the advisor's request to investigate the influence of mesh refinement
and discuss the spatial convergence of the *reference FE solution* --
this is a property of the CPU Newton-Raphson solver itself
(`omar_pfem/mesh_convergence.py`), independent of the neural operator or
`MESH_N`/`PROTOCOL` above.

Uses fixed, closed-form analytic (E, nu, load) fields instead of the
random GRF ensemble used everywhere else in this study, so the *exact
same* physical field can be solved at every resolution (the GRF sampler's
random phase array is sized by the mesh resolution itself, so re-sampling
it at a different N does not give a finer view of the same field -- see
`mesh_convergence.py`'s module docstring). This checks solver convergence for all three material models and both
geometries; `CONVERGENCE_MATERIALS` below now covers Neo-Hookean,
Mooney-Rivlin, and Arruda-Boyce. The resume logic (see
`mesh_convergence.py`'s `_load_done_rows`) means any resolution already
saved from an earlier, partial run (e.g. B2 x Neo-Hookean, previously
interrupted after N=26) is skipped and simply continues from where it
left off -- no separate step is needed to finish an incomplete run.


In [ ]:
CONVERGENCE_RESOLUTIONS = "6,11,16,21,26,31,41,51"   # node-count-per-side; 21 = this study's own mesh
CONVERGENCE_MATERIALS = ["neo_hookean", "mooney_rivlin", "arruda_boyce"]  # full coverage, all three material models
_requested_Ns = {int(n) for n in CONVERGENCE_RESOLUTIONS.split(",") if n.strip()}

convergence_dir = os.path.join(RESULTS_DIR, "mesh_convergence")
os.makedirs(convergence_dir, exist_ok=True)

for geometry in GEOMETRIES:
    for material in CONVERGENCE_MATERIALS:
        out_json = os.path.join(convergence_dir, f"{geometry}_{material}_convergence.json")
        # A partially-completed file (e.g. from an earlier interrupted run)
        # must NOT be treated as fully done just because the file exists --
        # only skip if every resolution actually requested is already saved.
        # mesh_convergence.py's own internal resume logic (_load_done_rows)
        # then picks up exactly where a partial file left off.
        already_complete = False
        if os.path.exists(out_json):
            with open(out_json) as f:
                _saved = _json.load(f)
            _saved_Ns = {row["N"] for row in _saved.get("rows", [])}
            already_complete = _requested_Ns.issubset(_saved_Ns)
        if already_complete:
            print(f"[{geometry}/{material}] convergence study already done -> {out_json}, skipping.")
            continue
        run_streaming([
            sys.executable, "-u", "-m", "omar_pfem.mesh_convergence",
            "--geometry", geometry, "--material", material,
            "--resolutions", CONVERGENCE_RESOLUTIONS,
            "--out_json", out_json,
        ], cwd=WORK_DIR, log_path=os.path.join(convergence_dir, f"{geometry}_{material}.log"))

print("\nMesh convergence study complete -- see", convergence_dir)


## Cell 10 - GPU-native FEM solver: correctness check + timing benchmark

Per the advisor's request for a GPU-native FEM solver and its cost
compared to the reference CPU solve (`NATIVE_FEM_SECONDS_PER_SAMPLE`
above). Two scripts, in order:

1. `validate_gpu_fem_solver.py` -- solves the SAME problem instance with
   both the existing CPU Newton-Raphson solver and the new
   `torch.func`-based batched GPU solver (`gpu_fem_solver.py`) and reports
   their agreement. Already verified locally to ~1e-13 relative error
   (machine precision) for both geometries and all 3 materials -- this
   just re-confirms that on the actual Colab GPU before trusting any
   timing number.
2. `gpu_fem_benchmark.py` -- only meaningful once every validation above
   reports PASS. Times BATCHED solves (multiple independent samples
   solved simultaneously via `torch.func.vmap`, the natural way to use a
   GPU) at several batch sizes, and reports ms/sample and the speedup vs.
   the 8.0 s/sample CPU reference.


In [ ]:
GPU_SOLVER_N = MESH_N          # same mesh resolution as the main study, for a like-for-like comparison
GPU_SOLVER_BATCH_SIZES = "1,8,32,128"

gpu_solver_dir = os.path.join(RESULTS_DIR, "gpu_fem_solver")
os.makedirs(gpu_solver_dir, exist_ok=True)

print("Step 1/2: correctness validation (GPU-style solver vs. CPU reference)")
validation_ok = True
for geometry in GEOMETRIES:
    for material in MATERIALS:
        log_path = os.path.join(gpu_solver_dir, f"validate_{geometry}_{material}.log")
        try:
            run_streaming([
                sys.executable, "-u", "-m", "omar_pfem.validate_gpu_fem_solver",
                "--geometry", geometry, "--material", material, "--N", "11",
            ], cwd=WORK_DIR, log_path=log_path)
        except RuntimeError as e:
            validation_ok = False
            print(f"[{geometry}/{material}] validation command failed: {e}")
            continue
        with open(log_path) as f:
            if "PASS" not in f.read():
                validation_ok = False
                print(f"[{geometry}/{material}] did NOT report PASS -- see {log_path}")

if not validation_ok:
    print("\nWARNING: at least one (geometry, material) did not PASS validation -- "
          "do NOT trust the timing numbers below until this is resolved.")
else:
    print("\nAll (geometry, material) pairs PASSED -- proceeding to timing benchmark.")

    print("\nStep 2/2: timing benchmark")
    for geometry in GEOMETRIES:
        for material in MATERIALS:   # timed for all three material models, not just neo_hookean
            out_json = os.path.join(gpu_solver_dir, f"{geometry}_{material}_timing.json")
            if os.path.exists(out_json):
                print(f"[{geometry}/{material}] GPU FEM timing already done -> {out_json}, skipping.")
                continue
            run_streaming([
                sys.executable, "-u", "-m", "omar_pfem.gpu_fem_benchmark",
                "--geometry", geometry, "--material", material, "--N", str(GPU_SOLVER_N),
                "--batch_sizes", GPU_SOLVER_BATCH_SIZES, "--n_repeats", "3",
                "--out_json", out_json,
            ], cwd=WORK_DIR, log_path=os.path.join(gpu_solver_dir, f"timing_{geometry}_{material}.log"))

print("\nGPU FEM solver correctness + timing complete -- see", gpu_solver_dir)


## Cell 11 - Out-of-distribution (OOD) evaluation

Per the advisor's request: "the test data are sampled from the same
distribution as the training data ... include OOD predictions, for
example different material ranges and loading magnitudes." For each of
the 6 already-trained cases (Cell 6/7): generates a small OOD-only test
set with material stiffness and load magnitude shifted ~2 std away from
the in-distribution mean used everywhere else in this study, then
evaluates that case's existing checkpoint on it -- **no retraining or
fine-tuning**. Requires Cell 6/7 to have produced a checkpoint first;
cases without one yet are skipped with a message.

Uses `--id_ntrain NTRAIN` when evaluating the in-distribution side --
`evaluate_ood.py` needs this to correctly read the SAME held-out test
slice (`samples[NTRAIN:NTRAIN+NTEST]`) the case was actually validated
on during training, rather than accidentally reading samples that were
part of the training set (see that script's module docstring for why
this matters).


In [ ]:
OOD_NTEST = 200
# In-distribution defaults (see data_generate_B{1,2}.py): E~N(1000,200), nu~N(0.3,0.05) clipped to
# [0.2,0.4], B1 load ty~N(-5,2), B2 pressure p~N(5,2). OOD shifts both material stiffness and load
# magnitude ~2 std away from those means -- "different material ranges and loading magnitudes" per
# the advisor's own wording, not an arbitrarily extreme shift.
OOD_E_MEAN, OOD_E_STD = 1500.0, 200.0
OOD_TY_MEAN, OOD_TY_STD = -9.0, 2.0    # B1
OOD_P_MEAN, OOD_P_STD = 9.0, 2.0       # B2

ood_data_dir = os.path.join(DATA_DIR, "ood")                          # local scratch: fast, disposable
ood_archive_dir = os.path.join(DATASETS_ARCHIVE_DIR, "ood")           # Drive: durable copy of each OOD NPZ
ood_results_dir = os.path.join(RESULTS_DIR, "ood_eval")               # Drive: reports/logs
os.makedirs(ood_results_dir, exist_ok=True)

for geometry, material in CASES:
    name = case_name(geometry, material)
    case_out_dir = os.path.join(RESULTS_DIR, name)
    checkpoint = os.path.join(case_out_dir, "model_best.pt")
    if not os.path.exists(checkpoint):
        checkpoint = os.path.join(case_out_dir, "model_final.pt")
    if not os.path.exists(checkpoint):
        print(f"[{name}] no checkpoint found yet -- run Cell 6/7 first, skipping OOD eval.")
        continue

    id_path = os.path.join(DATASETS_ARCHIVE_DIR, name, "hyperelastic_training_data_q4.npz")
    if not os.path.exists(id_path):
        print(f"[{name}] in-distribution dataset not found at {id_path}, skipping OOD eval.")
        continue

    out_json = os.path.join(ood_results_dir, f"{name}_ood_report.json")
    if os.path.exists(out_json):
        print(f"[{name}] OOD evaluation already done -> {out_json}, skipping.")
        continue

    ood_h5_dir = os.path.join(ood_data_dir, f"fem_{name}_ood")
    ood_npz_dir = os.path.join(ood_data_dir, f"npz_{name}_ood")
    ood_path = os.path.join(ood_npz_dir, "hyperelastic_training_data_q4.npz")
    ood_archive_path = os.path.join(ood_archive_dir, name, "hyperelastic_training_data_q4.npz")

    # Same durability pattern as ensure_dataset(): restore from Drive if a
    # disconnect wiped local scratch after this OOD set was already
    # generated, instead of regenerating it from scratch.
    if os.path.exists(ood_path):
        print(f"[{name}] OOD dataset already exists locally, skipping generation.")
    elif os.path.exists(ood_archive_path):
        print(f"[{name}] Restoring archived OOD dataset from Drive ({ood_archive_path}).")
        os.makedirs(ood_npz_dir, exist_ok=True)
        shutil.copy2(ood_archive_path, ood_path)
    else:
        if geometry == "B1":
            gen_module = "omar_pfem.data.data_generate_B1"
            mesh_args = ["--Nx", str(MESH_N), "--Ny", str(MESH_N)]
            shift_args = ["--ty_mean", str(OOD_TY_MEAN), "--ty_std", str(OOD_TY_STD)]
            conv_module = "omar_pfem.data.convert_B1_quad"
        else:
            gen_module = "omar_pfem.data.data_generate_B2"
            mesh_args = ["--Ntheta", str(MESH_N), "--Nr", str(MESH_N)]
            shift_args = ["--p_mean", str(OOD_P_MEAN), "--p_std", str(OOD_P_STD)]
            conv_module = "omar_pfem.data.convert_B2_quad"

        run_streaming([
            sys.executable, "-u", "-m", gen_module,
            "--num_index", "1", "--num_samples", str(OOD_NTEST),
            *mesh_args, "--material", material, "--n_workers", str(N_WORKERS),
            "--E_mean", str(OOD_E_MEAN), "--E_std", str(OOD_E_STD),
            *shift_args,
            "--out_dir", ood_h5_dir,
        ], cwd=WORK_DIR, log_path=os.path.join(ood_results_dir, f"{name}_ood_data_generation.log"))
        run_streaming([
            sys.executable, "-u", "-m", conv_module,
            "--h5_dir", ood_h5_dir, "--out_dir", ood_npz_dir,
        ], cwd=WORK_DIR, log_path=os.path.join(ood_results_dir, f"{name}_ood_npz_conversion.log"))

        os.makedirs(os.path.dirname(ood_archive_path), exist_ok=True)
        shutil.copy2(ood_path, ood_archive_path)
        print(f"[{name}] OOD dataset archived to {ood_archive_path}")

    run_streaming([
        sys.executable, "-u", "-m", "omar_pfem.evaluate_ood",
        "--geometry", geometry, "--material", material,
        "--checkpoint", checkpoint,
        "--id_path", id_path, "--id_ntrain", str(NTRAIN),
        "--ood_path", ood_path,
        "--ntest", str(OOD_NTEST), "--out_json", out_json,
    ], cwd=WORK_DIR, log_path=os.path.join(ood_results_dir, f"{name}_ood_eval.log"))

print("\nOOD evaluation complete -- see", ood_results_dir)


## Cell 12 - Resolution-invariance study (10 independent trainings across mesh sizes)

Per the advisor's request: "train on two different meshes and then
training on further ... meshes (altogether 10) and showing the results
are always the same." This trains the SAME protocol independently at
each of `RES_STUDY_RESOLUTIONS` and compares the resulting validation
errors side by side -- it does not need a new capability (Transolver is
mesh-size-agnostic and the data-generation scripts already take mesh
resolution as a CLI argument), just the orchestration driver
`resolution_invariance_study.py`.

**This is the single most expensive item in this notebook**: each
resolution repeats the full generate-dataset + train pipeline (up to
`CONTINUE_EPOCHS`, same early-stopping settings as the main 6 cases), so
the whole cell is roughly 10x the cost of training one case. By default
this runs for B1 x Neo-Hookean only (one geometry/material, matching the
scope of the mesh-convergence cell above); add entries to
`RES_STUDY_CASES` for more coverage if the compute budget allows it.

Resumes cleanly if interrupted, the same way the main 6 cases do: a
resolution that already finished (`model_final.pt`/`EARLY_STOPPED` under
its Drive-backed `--out_dir`) is skipped entirely -- its dataset is never
touched again, so a disconnect never forces regenerating an
already-finished resolution's data. A resolution that was only partway
done (dataset generated, training interrupted) restores its dataset from
`--archive_dir` (Drive) instead of regenerating it, exactly mirroring
`ensure_dataset`'s pattern for the 6 main cases. The summary table flags
any resolution that hit the epoch cap without early-stopping -- treat
those as lower bounds, not converged results.


In [ ]:
RES_STUDY_RESOLUTIONS = "13,17,21,25,29,33,37,41,45,49"   # 10 values; 21 = this study's own mesh
RES_STUDY_CASES = [("B1", "neo_hookean")]                 # add more (geometry, material) pairs if budget allows

res_study_data_dir = os.path.join(DATA_DIR, "resolution_study")            # local scratch: fast, disposable
res_study_archive_dir = os.path.join(DATASETS_ARCHIVE_DIR, "resolution_study")  # Drive: durable, per-resolution NPZs
res_study_results_dir = os.path.join(RESULTS_DIR, "resolution_study")      # Drive: checkpoints/metrics/summary
os.makedirs(res_study_results_dir, exist_ok=True)

for geometry, material in RES_STUDY_CASES:
    name = case_name(geometry, material)
    case_out_dir = os.path.join(res_study_results_dir, name)
    os.makedirs(case_out_dir, exist_ok=True)
    summary_path = os.path.join(case_out_dir, "resolution_study_summary.json")
    if os.path.exists(summary_path):
        print(f"[{name}] resolution-invariance study already done -> {summary_path}, skipping.")
        continue

    run_streaming([
        sys.executable, "-u", "-m", "omar_pfem.resolution_invariance_study",
        "--geometry", geometry, "--material", material,
        "--resolutions", RES_STUDY_RESOLUTIONS,
        "--ntrain", str(NTRAIN), "--ntest", str(NTEST), "--n_workers", str(N_WORKERS),
        "--batch_size", str(PROTOCOL["batch_size"]),
        "--epochs", str(PROTOCOL["epochs"]),
        "--validate_every", str(PROTOCOL["validate_every"]),
        "--early_stop_patience", str(PROTOCOL["early_stop_patience"]),
        "--early_stop_min_delta", str(PROTOCOL["early_stop_min_delta"]),
        "--data_dir", os.path.join(res_study_data_dir, name),
        "--archive_dir", os.path.join(res_study_archive_dir, name),
        "--out_dir", case_out_dir,
    ], cwd=WORK_DIR, log_path=os.path.join(res_study_results_dir, f"{name}.log"))

print("\nResolution-invariance study complete -- see", res_study_results_dir)


## Cell 13 - Zip and download all results (optional)

Results already live on Google Drive (`RESULTS_DIR`), so this is just a
convenience if you want a local zip copy too -- not required for safety.


In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive('/content/pfem_results', 'zip', RESULTS_DIR)
print(f"Archived {RESULTS_DIR} -> {archive_path}")
files.download(archive_path)
